In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

# ── Config ──────────────────────────────────────────────────────────────────
MAX_HISTORY_TURNS = 3
 
system_prompt = "You are a helpful assistant."

In [ ]:
# ── LLM setup ──────────────────────────────────────────────────────────────
_llm = ChatOpenAI(
    base_url    = os.getenv("LLM_BASE_URL"),
    api_key     = "EMPTY",
    model       = os.getenv("LLM_MODEL"),
    temperature = 0.05,
    max_tokens  = 4096,
    timeout     = 120,
    streaming=True
) 

In [ ]:
# ── Basic LLM Invoke ─────────────────────────────────────────────────
from IPython.display import Markdown, display

response = _llm.invoke([SystemMessage(content=system_prompt),
                        HumanMessage(content="write a code for a simple calculator")]).content

display(Markdown(response))

In [ ]:
# ── Core inference function ─────────────────────────────────────────────────
def chat(user_message: str, history: list):
    # Sliding window
    windowed_history = history[-MAX_HISTORY_TURNS:]
 
    messages = [SystemMessage(content=system_prompt)]
 
    for turn in windowed_history:
        # Gradio may pass dicts {"role":..,"content":..} or tuples (user, assistant)
        if isinstance(turn, dict):
            if turn["role"] == "user":
                messages.append(HumanMessage(content=turn["content"]))
            elif turn["role"] == "assistant":
                messages.append(AIMessage(content=turn["content"]))
        else:
            human, assistant = turn
            messages.append(HumanMessage(content=human))
            if assistant:
                messages.append(AIMessage(content=assistant))
 
    messages.append(HumanMessage(content=user_message))
 
    partial = ""
    for chunk in _llm.stream(messages):
        partial += chunk.content
        yield partial

In [ ]:
# ── Gradio ChatInterface ────────────────────────────────────────────────────
demo = gr.ChatInterface(
    fn          = chat,
    title       = "🚀 vLLM Chat",
    description = "Chat powered by vLLM inference",
)
 
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860, share=False)
